<a href="https://colab.research.google.com/github/vishal6975/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishal6975/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I use Logistic Regression because my target is binary and the model is simple and interpretable.

My lane is CTR review. The model uses observed search signals such as impressions, clicks, CTR, and average position.

I chose a simple model first so I can clearly compare it with my Week-4 rule-based baseline. The goal is to see whether the model adds useful signal, not to reward complexity.

In [12]:
print("Method: Logistic Regression")
print("Reason: binary target, simple, interpretable, and suitable for comparison with the rule-based baseline.")

Method: Logistic Regression
Reason: binary target, simple, interpretable, and suitable for comparison with the rule-based baseline.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a time-aware split.

Earlier March 2026 observations are used as model inputs and later March 2026 observations are used for the outcome.

This avoids using future performance as a feature. The model and Week-4 baseline are evaluated on the same evaluation rows and with the same metric.

In [13]:
print("Split: time-aware")
print("Features: earlier March 2026")
print("Outcome: later March 2026")
print("Future information used as features: NO")

Split: time-aware
Features: earlier March 2026
Outcome: later March 2026
Future information used as features: NO


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I train Logistic Regression using early-period search signals and compare its F1 score with the Week-4 rule-based baseline.

F1 is used because it balances precision and recall for the binary review outcome. Both methods are evaluated on the same rows.



To securely use your Hugging Face token, add it to Colab's Secrets manager.

1.  Click the "🔑" icon in the left-hand panel.
2.  Click "+ New secret".
3.  Set the "Name" to `HF_TOKEN`.
4.  Paste your token in the "Value" field.
5.  Ensure "Notebook access" is enabled.

After setting the secret, run the following cell to make the token available as an environment variable.

In [14]:
# Import the userdata module from google.colab
from google.colab import userdata
import os

# Get the HF_TOKEN from Colab's secrets
# This assumes you have added a secret named 'HF_TOKEN' in the Colab secrets manager
try:
    hf_token_value = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token_value
    print("HF_TOKEN successfully loaded from Colab Secrets.")
except userdata.SecretError:
    print("Error: HF_TOKEN not found in Colab Secrets. Please add it via the 🔑 icon.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# Verify if the token is set (optional)
# if 'HF_TOKEN' in os.environ:
#     print("HF_TOKEN is set in environment variables.")
# else:
#     print("HF_TOKEN is NOT set in environment variables.")

HF_TOKEN successfully loaded from Colab Secrets.


In [15]:
import duckdb, os, numpy as np, pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

# Connect to DuckDB
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Use the Hugging Face token already stored in Colab
token = os.environ.get("HF_TOKEN")

if not token:
    raise RuntimeError("HF_TOKEN not found. Run your Hugging Face login cell first.")

con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{token}'
);
""")

path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Load required data
df = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet('{path}')
""").df()

df["ctr"] = np.where(
    df.gsc_impressions > 0,
    df.gsc_clicks / df.gsc_impressions,
    np.nan
)

# Early March = features
early = df[df.report_date <= "2026-03-15"].groupby(
    ["client_hash_id", "content_hash_id"]
).agg(
    impressions_early=("gsc_impressions", "sum"),
    clicks_early=("gsc_clicks", "sum"),
    ctr_early=("ctr", "mean"),
    avg_position_early=("gsc_avg_position", "mean")
).reset_index()

# Late March = outcome
late = df[df.report_date >= "2026-03-16"].groupby(
    ["client_hash_id", "content_hash_id"]
).agg(
    ctr_late=("ctr", "mean")
).reset_index()

model_df = early.merge(
    late,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Target
model_df["future_ctr_declined"] = (
    model_df["ctr_late"] < model_df["ctr_early"]
).astype(int)

features = [
    "impressions_early",
    "clicks_early",
    "ctr_early",
    "avg_position_early"
]

X = model_df[features].replace([np.inf, -np.inf], np.nan)
y = model_df["future_ctr_declined"]

valid = X.notna().all(axis=1)

X = X[valid]
y = y[valid]

# Train model
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )
)

model.fit(X, y)

pred = model.predict(X)

print("MODEL COMPLETE")
print("=" * 50)
print("Rows:", len(X))
print("Precision:", round(precision_score(y, pred, zero_division=0), 4))
print("Recall:", round(recall_score(y, pred, zero_division=0), 4))
print("F1:", round(f1_score(y, pred, zero_division=0), 4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

MODEL COMPLETE
Rows: 151980
Precision: 0.7114
Recall: 0.7191
F1: 0.7152


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model errors are reviewed using false positives and false negatives.

False positives are pages predicted for review that did not show the later CTR decline. False negatives are pages that experienced the decline but were not predicted.

The model mainly relies on the observed search-performance signals supplied as features. These signals are directional and may be affected by search intent, SERP features, and normal CTR variation.

Therefore, the model is decision-support and not a guarantee that a page requires an action.

In [16]:
errors = pd.DataFrame({
    "actual": y,
    "predicted": pred
})

print("False positives:",
      ((errors.actual == 0) & (errors.predicted == 1)).sum())

print("False negatives:",
      ((errors.actual == 1) & (errors.predicted == 0)).sum())

coef = model.named_steps["logisticregression"].coef_[0]

print("\nFeature coefficients:")
print(pd.Series(coef, index=features).sort_values())

False positives: 10412
False negatives: 10025

Feature coefficients:
avg_position_early   -0.405481
impressions_early     0.353550
clicks_early          2.881308
ctr_early             3.405150
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.